<a href="https://www.kaggle.com/code/pcsciprav/qe-guide-main-branch-quantum-espresso-on-gpu?scriptVersionId=372918089" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Quantum ESPRESSO installation guide on Python for Dual T4 GPUs

##  1. Install Dependencies

In [ ]:
%%bash
apt-get update
apt-get install -y build-essential wget \
  libfftw3-dev libblas-dev liblapack-dev \
  libscalapack-mpi-dev libopenmpi-dev openmpi-bin
apt-get install -y libxc-dev libxc9
pip install numpy scipy pyyaml matplotlib phonopy ase cclib gpu4pyscf-cuda12x

##  2. Download Quantum ESPRESSO and NVFORTRAN/NVHPC SDK

Will take ~15-17 GiB of output space, ensure there is enough storage

In [ ]:
%%bash
cd 
wget https://github.com/QEF/q-e/archive/refs/tags/qe-7.6.tar.gz
tar xpzf qe-7.6.tar.gz
rm -rf /kaggle/working/qe-7.6.tar.gz
cd q-e-qe-7.6

Here, nvfortran is a requirement otherwise the code cannot work. Use cuda 12.3 or above (use versions which give direct tarball over the newer 13.0+ interactive installer as it does not work).

In [ ]:
%%bash
cd /kaggle/working
wget https://developer.download.nvidia.com/hpc-sdk/24.11/nvhpc_2024_2411_Linux_x86_64_cuda_12.6.tar.gz
tar xpzf nvhpc_2024_2411_Linux_x86_64_cuda_12.6.tar.gz
rm -rf nvhpc_2024_2411_Linux_x86_64_cuda_12.6.tar.gz

## 3. Setting environment for running

This part is extremely crucial, **do not** skip. Without this step, the whole process is not applicable.

In [1]:
import os

base = "/root/Linux_x86_64/26.11" # Change path to current path where the nvhpc folder is
os.environ["PATH"] = base + "/compilers/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = base + "/compilers/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
print("Environment set.")
!nvfortran --version
print(" ")
print("Check for error")

Environment set.

nvfortran 26.11-0 64-bit target on x86-64 Linux -tp znver3 
NVIDIA Compilers and Tools
Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES.  All rights reserved.
 
Check for error


## 4. Set config

Ensure to not tinker with the config unless **absolutely needed**. **Do not modify.** Change the --with-cuda-runtime and --with-cuda-cc as per your CUDA version and compute capability(90 for Hopper, 80 for Ampere, 75 for Turing, 52 for Maxwell,

In [2]:
%%bash
cd ~/q-e-qe-7.6 # Change path as per need
# Change according to the compute capability running nvidia-smi shows (if it shows as 7.5 or 10.0 multiply by 10)
export LIBXC_LIBS="-L/usr/lib/x86_64-linux-gnu -lxc -lxcf03"
export LIBXC_CFLAGS="-I/usr/include"
./configure \
  FC=nvfortran \
  F90=nvfortran \
  F77=nvfortran \
  MPIF90=nvfortran \
  CC=nvc \
  CXX=nvc++ \
  LD=nvfortran \
  --with-gpu=cuda \
  --with-cuda-runtime=13.4 \
  --with-cuda-cc=80 \
  --with-scalapack=no

in-source build
checking build system type... x86_64-pc-linux-gnu
checking ARCH... x86_64
checking setting AR... ... ar
checking setting ARFLAGS... ... ruv
checking whether the Fortran compiler works... yes
checking for Fortran compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether we are using the GNU Fortran compiler... no
checking whether nvfortran accepts -g... yes
checking for Fortran flag to compile .f90 files... none
checking for nvfortran... nvfortran
checking whether we are using the GNU Fortran compiler... no
checking whether nvfortran accepts -g... yes
checking version of nvfortran... nvfortran 26.11-0


configure: WARNING: Check whether user-supplied F90 is consistent with MPIF90 !!! 


setting F90... nvfortran
setting MPIF90... nvfortran
checking whether we are using the GNU C compiler... yes
checking whether nvc accepts -g... yes
checking for nvc option to accept ISO C89... none needed
setting CC... nvc
setting CFLAGS... -O3
using F90... nvfortran
setting FFLAGS... -fast
setting F90FLAGS... -fast -Mcache_align -Mpreprocess -Mlarge_arrays
setting FFLAGS_NOOPT... -O0
setting FFLAGS_NOMAIN... -Mnomain
setting CPP... cpp
setting CPPFLAGS... -P -traditional -Uvector
setting LD... nvfortran
setting LDFLAGS...
checking whether Fortran compiler accepts -cuda -gpu=cuda13.4... yes
checking whether make sets $(MAKE)... yes
checking whether Fortran files must be preprocessed... no
checking for library containing dgemm... no
MKL not found
in /opt/intel/mkl/lib/intel64: checking for library containing dgemm... no
MKL not found
in /root/Linux_x86_64/26.11/compilers/lib: checking for library containing dgemm... no
MKL not found
checking for library containing dgemm... no
in /usr/lo

If final output shows `configure: success`, continue. Otherwise, search for documentation or use AI to help troubleshoot.

In [3]:
from pathlib import Path
import re

path = Path("/root/q-e-qe-7.6/make.inc")
text = path.read_text()

new_lines = []
for line in text.splitlines():
    if line.strip().startswith("DFLAGS"):
        line = re.sub(r"-D__DFTI\b", "", line)
        line = re.sub(r"-D__FFTW3\b", "", line)
        line = re.sub(r"-D__FFTW\b", "", line)
        line = line.rstrip() + " -D__FFTW"
    new_lines.append(line)

path.write_text("\n".join(new_lines) + "\n")

print("Updated DFLAGS:")
for line in new_lines:
    if line.strip().startswith("DFLAGS"):
        print(line)

Updated DFLAGS:
DFLAGS         =  -D__PGI -D__CUDA -D__FFTW


## 5. Compile and Run

The below code may take a few minutes to run. **Do not** interrupt execution. In case of errors saying certain parts are not there/broken, run `!make distclean`, re-configure(using the `./configure` block), and try again. If it fails again, restart the container.

In [4]:
!cd /root/q-e-qe-7.6/ && make pw -j$(nproc) 2>&1

ar ruv libdavid_rci.a david_rci.o       
ar: `u' modifier ignored since `D' is the default (see `U')
ar: creating libdavid_rci.a
a - david_rci.o
ranlib libdavid_rci.a    
make[2]: Leaving directory '/root/q-e-qe-7.6/KS_Solvers/Davidson_RCI'
if test -d Davidson ; then \
        ( cd Davidson ; make all || exit 1 ) ; fi
make[2]: Entering directory '/root/q-e-qe-7.6/KS_Solvers/Davidson'
nvfortran -fast -Mcache_align -Mpreprocess -Mlarge_arrays -D__PGI -D__CUDA -D__FFTW  -cuda -gpu=cc80,cuda13.4 -acc -I/root/q-e-qe-7.6/devxlib/src -I/root/q-e-qe-7.6/external/devxlib/include -I. -I/root/q-e-qe-7.6/include -I ../../LAXlib -I ../../UtilXlib -I. -c cegterg.f90 -o cegterg.o
nvfortran -fast -Mcache_align -Mpreprocess -Mlarge_arrays -D__PGI -D__CUDA -D__FFTW  -cuda -gpu=cc80,cuda13.4 -acc -I/root/q-e-qe-7.6/devxlib/src -I/root/q-e-qe-7.6/external/devxlib/include -I. -I/root/q-e-qe-7.6/include -I ../../LAXlib -I ../../UtilXlib -I. -c regterg.f90 -o regterg.o
nvfortran -fast -Mcache_align -Mpreproc

This code creates the QE executable file for mpi

In [19]:
%%bash
cat > run_pw.sh << 'EOF'
#!/bin/bash
export CUDA_VISIBLE_DEVICES=${OMPI_COMM_WORLD_LOCAL_RANK}
exec /root/q-e-qe-7.6/bin/pw.x -nb 8 -nk 2 -in input.in > output_o2.out
EOF
chmod +x run_pw.sh

Below code runs QE and tails it continuously until ```mpirun``` concludes.

In [20]:
!mpirun --allow-run-as-root -np 1 --bind-to none ./run_pw.sh & tail -n 20 --pid=$! -F output_o2.out

tail: cannot open 'output_o2.out' for reading: No such file or directory


tail: 'output_o2.out' has appeared;  following new file

     Program PWSCF v.7.6 starts on 27Nov2026 at  4:26: 5 

     This program is part of the open-source Quantum ESPRESSO suite
     for quantum simulation of materials; please cite
         "P. Giannozzi et al., J. Phys.:Condens. Matter 21 395502 (2009);
         "P. Giannozzi et al., J. Phys.:Condens. Matter 29 465901 (2017);
         "P. Giannozzi et al., J. Chem. Phys. 152 154105 (2020);
          URL http://www.quantum-espresso.org", 
     in publications or presentations arising from this work. More details at
     http://www.quantum-espresso.org/quote

     Serial version
     156972 MiB available memory on the printing compute node when the environment starts
 
     Reading input from input.in

     Current dimensions of program PWSCF are:
     Max number of different atomic species (ntypx) = 10
     Max number of k-points (npk) =  40000
     Max angular momentum in pseudopotentials (lmaxx) =  4
     file O.pbe-n-kjpaw_psl

Check outputs for any errors (saves a lot of time compared to trying to use mpirun failure codes which are random and do not make sense). Please use ```!tail -n(number of lines) output.out``` for seeing the last N lines of the output if the output is large or only some parts needed.

In [ ]:
!cat output_graphite.out

## 6.(Optional) In case .in files not present and only .xyz file available

Run this if you have .in files and need ld1.x to run to create a .upf file. Ideally, use existing pseudos to reduce any errors, but if it's not available, generate your own pseudos. Please be warned, it comes with a fairly **major** risk of configs being wrong.

In [ ]:
%%bash
cd /kaggle/working/q-e-qe-7.6/ && make ld1 -j$(nproc)
export ELEMENT="U" # Change element name here
/kaggle/working/q-e-qe-7.6/bin/ld1.x < /kaggle/input/datasets/pcsciprav/actinide-basis-set-rel/fully-relativistic-basis/"$ELEMENT".in > "$ELEMENT".out
# Change the .in and .out file names to match the input .in file name. Use the same .out name.

In [ ]:
%%bash
cd /kaggle/working
cat << 'EOF' > Rg_pseudo.in
&input
    title='Rg semicore PAW',
    zed=111.0,
    rel=2,
    config='[Xe] 4f14 5d10 5f14 6s2 6p6 6d10 7s1 7p0',
    dft='PBE',
    iswitch=3,
/
&inputp
    lpaw=.true.,
    pseudotype=3,
    file_pseudopw='Rg.UPF',
    lloc=-1,
    rcloc=2.0,
    nlcc=.true.,
    rcore=1.5,
    which_augfun='PSQ',
/
8
6S  1  0  2.00  0.00  2.30  2.30  0.5
7S  2  0  1.00  0.00  2.40  2.40  0.5
6P  2  1  2.00  0.00  2.30  2.30  0.5
6P  2  1  4.00  0.00  2.30  2.30  1.5
7P  3  1  0.00  0.00  2.40  2.40  0.5
7P  3  1  0.00  0.00  2.40  2.40  1.5
6D  3  2  4.00  0.00  2.40  2.40  1.5
6D  3  2  6.00  0.00  2.40  2.40  2.5
EOF

Change the link to any pseudopotential file the script can't find.

In [ ]:
!wget https://pseudopotentials.quantum-espresso.org/upf_files/N.rel-pbe-n-kjpaw_psl.0.1.UPF
!wget https://pseudopotentials.quantum-espresso.org/upf_files/F.rel-pbe-n-kjpaw_psl.0.1.UPF
!wget https://pseudopotentials.quantum-espresso.org/upf_files/H.rel-pbe-kjpaw_psl.1.0.0.UPF

Run this snippet first and ensure persistence is off as persistence, as even with using `chmod +x`, Kaggle's environment breaks.

In [22]:
import os
import shutil
import requests

PSEUDO_DIR = "pseudo"
os.makedirs(PSEUDO_DIR, exist_ok=True)
BASE_URL = "https://pseudopotentials.quantum-espresso.org/upf_files/"

manual_pseudos = {
    #'U': '/kaggle/working/U.rel-pbe-spn-kjpaw_MS.UPF',
    #'F': '/kaggle/working/F.rel-pbe-n-kjpaw_psl.0.1.UPF',
    #'N': '/kaggle/working/N.rel-pbe-n-kjpaw_psl.0.1.UPF',
    #'H': '/kaggle/working/H.rel-pbe-kjpaw_psl.1.0.0.UPF'
}

masses = {
    'U': 238.03,
    'Pu': 244.06,
    'Au': 196.966570,
    'Fr': 223,
    'Cs': 132.91,
    'N': 14.007,
    'Nd': 144.242,
    'Tc': 97.907,
    'As': 74.9216,
    'O': 16,
    'C': 12,
    'Os': 190.23,
    'Pt': 198.9705762,
    'Hg': 200.59,
    'F': 18.998,
    'Cu': 63.546,
    'H': 1.008
}

hubbard_u = {
    #'U': 4.0
}

def find_pseudo(element):
    return [
        f"{element}.pbe-spn-kjpaw_psl.1.0.0.UPF",
        f"{element}.pbe-n-kjpaw_psl.1.0.0.UPF",
        f"{element}.pbe-kjpaw_psl.1.0.0.UPF",
        f"{element}.rel-pbe-spfn-kjpaw_psl.1.0.0.UPF", # Future self note, add if logic to check for Th and above and modify all to full rel by using heaviest to lightest logic
    ]

def try_download(element):
    for name in find_pseudo(element):
        path = os.path.join(PSEUDO_DIR, name)

        if os.path.exists(path):
            print(f"Using existing pseudo: {path}")
            return name

        url = BASE_URL + name
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200 and len(r.content) > 0:
                with open(path, "wb") as f:
                    f.write(r.content)
                print(f"Downloaded {name}")
                return name
        except Exception as e:
            print(f"Could not download {name}: {e}")

    return None

def install_manual_pseudo(element):
    src = manual_pseudos[element]
    if not os.path.exists(src):
        raise FileNotFoundError(f"Manual pseudopotential for {element} not found: {src}")

    filename = os.path.basename(src)
    dst = os.path.join(PSEUDO_DIR, filename)

    if not os.path.exists(dst):
        shutil.copyfile(src, dst)
        print(f"Copied manual pseudo for {element} to {dst}")
    else:
        print(f"Using existing copied pseudo for {element}: {dst}")

    return filename

def download_pseudo(element):
    if element in manual_pseudos:
        return install_manual_pseudo(element)

    result = try_download(element)
    if result:
        return result

    raise Exception(f"No pseudopotential found for {element}")

def xyz_to_qe(xyz_file, output="input.in"):
    with open(xyz_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    nat = int(lines[0].strip())
    coords = lines[2:2 + nat]

    atoms = []
    species_ordered = []

    for line in coords:
        parts = line.split()
        if len(parts) < 4:
            raise ValueError(f"Invalid XYZ line: {line!r}")
        el, x, y, z = parts[:4]
        x = float(x) + 10.0
        y = float(y) + 10.0
        z = float(z) + 10.0
        atoms.append((el, x, y, z))
        if el not in species_ordered:
            species_ordered.append(el)

    pseudo_map = {}
    for el in species_ordered:
        pseudo_map[el] = download_pseudo(el)

    with open(output, "w", encoding="ascii", newline="\n") as f:
        # CONTROL
        f.write("&CONTROL\n")
        f.write("  calculation = 'scf'\n") # Either keep it as relax or scf
        f.write("  restart_mode='from_scratch'\n") # Change later to restart
        f.write("  pseudo_dir = './pseudo/'\n")
        f.write("  outdir = './tmp/'\n")
        #f.write("  nstep = 200\n") # Comment out if not using relax
        f.write("  prefix = 'calc'\n")
        f.write("  tprnfor = .true.\n") # Remove this and the line below if forces for phonons not needed
        f.write("  tstress = .false.\n") # Dont use unless using vc-relax, otherwise it wastes calc time
        f.write("/\n\n")

        # SYSTEM
        f.write("&SYSTEM\n")
        f.write(f"  nat = {nat}\n")
        f.write(f"  ntyp = {len(species_ordered)}\n")
        f.write("  ecutwfc = 60\n")
        f.write("  starting_magnetization(1) = 1\n")
        #f.write("  starting_magnetization(2) = 0.01\n")
        #f.write("  starting_magnetization(3) = 0.2\n")'''
        #f.write("  tot_charge = 1\n") # Remove comment if using charged body
        f.write("  ecutrho = 720\n") # Ideally you want 8x or 12x of ecutwfc though due to VRAM limits I'm forced to use standard 4x
        f.write("  nspin = 2\n") # Only use for ones with known spin alongside the next line for known magnetisation, nspin 2 for open-shell, don't use nspin 4 it's a bad practice instead use noncolin
        #f.write("  tot_magnetization = 2\n")
        #f.write("  nbnd = 170\n")
        f.write("  input_dft = 'PBE'\n")
        f.write("  occupations = 'smearing'\n") # Might as well use fixed if you use really small smearing
        f.write("  smearing = 'gaussian'\n") # If fixed occupations, comment this and degauss line
        f.write("  degauss = 0.005\n")
        f.write("  vdw_corr = 'grimme-d3'\n")
        f.write("  dftd3_version = 4\n")
        f.write("  dftd3_threebody = .false.\n")
        f.write("  assume_isolated = 'mt'\n") # Or use makov-payne (depends on you), though it's untested in terms of VRAM saves
        f.write("  ibrav = 0\n")
        f.write("  nosym = .true.\n")
        f.write("  noinv = .true.\n")
        f.write("  noncolin = .false.\n")
        f.write("  lspinorb = .false.\n")
        f.write("/\n\n")

        # ELECTRONS
        f.write("&ELECTRONS\n")
        f.write("  conv_thr = 1.0d-8\n")
        f.write("  mixing_beta = 0.5\n") # Higher for more stable systems, less for unstable systems
        f.write("  mixing_mode = 'local-TF'\n")
        f.write("  electron_maxstep = 250\n")
        f.write("/\n\n")
        
        # IONS
        # Use only when running calculation = relax, otherwise comment it out.
        '''f.write("&IONS\n")
        f.write("  ion_dynamics = 'fire'\n") # Change to fire for faster convergance if bfgs fails but if fire fails use damp.
        f.write("  fire_f_inc = 1.6\n")
        f.write("  trust_radius_ini = 0.10\n") # Lower for unstable systems
        f.write("  trust_radius_max = 0.80\n") # Lower for unstable systems
        #f.write("  upscale = 0.5\n")
        f.write("/\n\n")'''
        # If using vc-relax, you need a &CELL namelist, which isnt here as I'm only doing for isolated molecules
        # ATOMIC_SPECIES
        f.write("ATOMIC_SPECIES\n")
        for el in species_ordered:
            mass = masses.get(el, 1.0)
            pseudo_filename = pseudo_map[el]
            f.write(f"{el:>3}  {mass:10.5f}  {pseudo_filename}\n")
        
        # ATOMIC_POSITIONS
        f.write("\nATOMIC_POSITIONS angstrom\n")
        for el, x, y, z in atoms:
            f.write(f"{el:>3}  {x}  {y}  {z}\n")

        
        # CELL PARAMETERS ANGSTROMS
        f.write("\nCELL_PARAMETERS angstrom\n")
        f.write("20.0 0.0 0.0\n")
        f.write("0.0 20.0 0.0\n")
        f.write("0.0 0.0 20.0\n")

        # K_POINTS
        f.write("\nK_POINTS gamma\n") # For SOC/noncolin, you have to use K_POINTS automatic and uncomment the below line
        #f.write("2 2 2 0 0 0\n")

        # HUBBARD
        has_hubbard = any(el in hubbard_u for el in species_ordered)
        if has_hubbard:
            f.write("\nHUBBARD ortho-atomic\n")
            for el in species_ordered:
                if el in hubbard_u:
                    f.write(f"U {el}-5f {hubbard_u[el]}\n")

    print(f"QE input ready: {output}")
    print("Pseudopotentials used:")
    for el in species_ordered:
        print(f"  {el} -> {pseudo_map[el]}")

xyz_to_qe("/root/C2N28O20_relaxed_shifted.xyz")

Using existing pseudo: pseudo/N.pbe-n-kjpaw_psl.1.0.0.UPF
Using existing pseudo: pseudo/C.pbe-n-kjpaw_psl.1.0.0.UPF
Using existing pseudo: pseudo/O.pbe-n-kjpaw_psl.1.0.0.UPF
QE input ready: input.in
Pseudopotentials used:
  N -> N.pbe-n-kjpaw_psl.1.0.0.UPF
  C -> C.pbe-n-kjpaw_psl.1.0.0.UPF
  O -> O.pbe-n-kjpaw_psl.1.0.0.UPF


Add fast files for reducing energies to find true energies in the RocketCEA/Cantera code

In [ ]:
%%bash
touch graphite.xyz
cat > graphite.xyz << 'EOF'
1
XYZ file generated by Avogadro.
C      1.0000000000    1.0000000000    1.0000000000
EOF

## 7. Phonons

### 7a. Full dynamical matrix method

In [23]:
!phonopy-init -d --qe --dim="1 1 1" -c input.in

        _
  _ __ | |__   ___  _ __   ___   _ __  _   _
 | '_ \| '_ \ / _ \| '_ \ / _ \ | '_ \| | | |
 | |_) | | | | (_) | | | | (_) || |_) | |_| |
 | .__/|_| |_|\___/|_| |_|\___(_) .__/ \__, |
 |_|                            |_|    |___/
                                       4.8.1

-------------------------[time 2026-11-27 04:30:30]-------------------------
Rust backend (phonors) using rayon (23 threads).
Python version 3.10.12
Spglib version 2.7.0

Calculator interface: qe
Crystal structure was read from "input.in".
Unit of length: au
Displacements creation mode
  Systematic displacements
Settings:
  Supercell: [1 1 1]
  Primitive matrix:
    [1. 0. 0.]
    [0. 1. 0.]
    [0. 0. 1.]
Spacegroup: P1 (1)
Number of symmetry operations in supercell: 1
Use -v option to watch primitive cell, unit cell, and supercell structures.

Generated number of supercells: 138
"phonopy_disp.yaml" and supercells have been created.

Summary of calculation was written in "phonopy_disp.yaml".
--------------

In [24]:
import glob

# 1. Read your baseline file and split it in half around the coordinates
with open('input.in', 'r') as f:
    baseline_lines = f.readlines()

pre_positions = []
post_positions = []
is_positions_block = False
passed_positions = False

# These keywords tell the script when the position block ends
qe_keywords = ['&CONTROL', '&SYSTEM', '&ELECTRONS', '&IONS', '&CELL', 
               'K_POINTS', 'CELL_PARAMETERS', 'ATOMIC_SPECIES']

for line in baseline_lines:
    line_upper = line.strip().upper()
    
    if line_upper.startswith('ATOMIC_POSITIONS'):
        is_positions_block = True
        continue
        
    if is_positions_block:
        if any(line_upper.startswith(kw) for kw in qe_keywords):
            is_positions_block = False
            passed_positions = True
            post_positions.append(line)
        continue
        
    if not is_positions_block:
        if not passed_positions:
            pre_positions.append(line)
        else:
            post_positions.append(line)

# 2. Extract the nudged coordinates from Phonopy's files
for target_file in glob.glob('supercell-*.in'):
    with open(target_file, 'r') as f:
        target_lines = f.readlines()
        
    new_positions = []
    in_target_pos = False
    
    for line in target_lines:
        line_upper = line.strip().upper()
        
        if line_upper.startswith('ATOMIC_POSITIONS'):
            in_target_pos = True
            new_positions.append(line)
            continue
            
        if in_target_pos:
            if any(line_upper.startswith(kw) for kw in qe_keywords):
                break
            if line.strip(): # Keep the coordinate lines
                new_positions.append(line)

    # 3. Sandwich the new coordinates inside your original baseline settings
    if new_positions:
        with open(target_file, 'w') as f:
            f.writelines(pre_positions)
            f.write("\n")
            f.writelines(new_positions)
            f.write("\n")
            f.writelines(post_positions)
        print(f"Successfully applied baseline settings to: {target_file}")
    else:
        print(f"Skipped: Could not find coordinates in {target_file}")

Successfully applied baseline settings to: supercell-074.in
Successfully applied baseline settings to: supercell-072.in
Successfully applied baseline settings to: supercell-101.in
Successfully applied baseline settings to: supercell-001.in
Successfully applied baseline settings to: supercell-002.in
Successfully applied baseline settings to: supercell-003.in
Successfully applied baseline settings to: supercell-004.in
Successfully applied baseline settings to: supercell-005.in
Successfully applied baseline settings to: supercell-006.in
Successfully applied baseline settings to: supercell-007.in
Successfully applied baseline settings to: supercell-008.in
Successfully applied baseline settings to: supercell-009.in
Successfully applied baseline settings to: supercell-010.in
Successfully applied baseline settings to: supercell-011.in
Successfully applied baseline settings to: supercell-012.in
Successfully applied baseline settings to: supercell-013.in
Successfully applied baseline settings t

Runs phonon calculations once phonpy has made the supercells.

In [ ]:
import glob
import re

# Set these per session. Each chunk is a separate run of the notebook.
START = 1
END = 111
LIMIT_SECONDS = 10 * 3600   # stop starting new jobs after this

all_files = glob.glob('supercell-*.in')
all_files.sort(key=lambda f: int(re.sub(r'\D', '', f)))

target_files = []
for f in all_files:
    match = re.search(r'-(\d+)\.', f)
    if match and START <= int(match.group(1)) <= END:
        target_files.append(f)

# Forces parser, run once per finished job
parser = '''import sys
import json
out = sys.argv[1]
txt = open(out).read()
if "JOB DONE" not in txt:
    sys.exit(1)
blocks = txt.split("Forces acting on atoms")
if len(blocks) < 2:
    sys.exit(1)
forces = []
for line in blocks[-1].splitlines():
    s = line.strip()
    if s.startswith("atom") and "force" in s:
        p = s.split()
        forces.append([float(p[-3]), float(p[-2]), float(p[-1])])
    elif forces and not s:
        break
print(json.dumps({"name": out, "forces": forces}))
'''
with open("parse_forces.py", "w") as f:
    f.write(parser)

worker = '''#!/bin/bash
export CUDA_VISIBLE_DEVICES=${OMPI_COMM_WORLD_LOCAL_RANK}
RANK=${OMPI_COMM_WORLD_LOCAL_RANK}
SIZE=${OMPI_COMM_WORLD_LOCAL_SIZE}
T0=$(date +%s)
FILES=("$@")

for (( i=$RANK; i<${#FILES[@]}; i+=$SIZE )); do
    IN_FILE=${FILES[$i]}
    OUT_FILE="${IN_FILE%.in}.out"

    if [ -f "$OUT_FILE" ] && grep -q "JOB DONE" "$OUT_FILE"; then
        echo "Rank $RANK: skipping $IN_FILE, already done"
        continue
    fi

    if (( $(date +%s) - T0 > __LIMIT__ )); then
        echo "Rank $RANK: time limit reached, stopping"
        break
    fi

    WORKDIR="work_rank_${RANK}_idx_${i}"
    mkdir -p $WORKDIR && cp $IN_FILE $WORKDIR/in.in

    echo "Rank $RANK: starting $IN_FILE on GPU $RANK"
    /root/q-e-qe-7.6/bin/pw.x -in $WORKDIR/in.in > "$OUT_FILE.tmp"

    if grep -q "JOB DONE" "$OUT_FILE.tmp"; then
        mv "$OUT_FILE.tmp" "$OUT_FILE"
        python3 parse_forces.py "$OUT_FILE" >> forces_rank_${RANK}.jsonl
    else
        echo "Rank $RANK: $IN_FILE did not finish"
        rm -f "$OUT_FILE.tmp"
    fi
    rm -rf $WORKDIR
done
'''
with open("run_parallel.sh", "w") as f:
    f.write(worker.replace("__LIMIT__", str(LIMIT_SECONDS)))

!chmod +x run_parallel.sh

files_str = " ".join(target_files)
!mpirun --allow-run-as-root -np 1 --bind-to none ./run_parallel.sh {files_str}

FORTRAN STOP
Rank 0: starting supercell-002.in on GPU 0
FORTRAN STOP
Rank 0: starting supercell-003.in on GPU 0


Use this cell for post-run cleanup

In [ ]:
!rm -rf supercell-*.in
!rm -rf supercell.in

In [ ]:
!phonopy --qe -f supercell-[0-9][0-9][0-9].out

In [ ]:
!touch mesh.conf
'''import os
os.system('locate "*.upf" > upf.txt')
upf = []
with open('upf.txt', 'r') as file:
        for line in file:
            line_array = line.strip().split() 
            upf.append(line_array)
            upf.removeprefix("/kaggle/working/q-e-qe-7.6/pseudo")
            upf.removesuffix(".pbe-n-kjpaw_psl.1.0.0.UPF")
'''
with open("mesh.conf", "w") as f:
    f.write("ATOM_NAME = C F N O\n") # Future me, add the commented logic and make it work(currently very broken)
    f.write("DIM = 1 1 1\n")
    f.write("MESH = 1 1 1\n")
    f.write("FORCE_SETS = READ\n")

print("mesh.conf has been created successfully.")

In [ ]:
!phonopy --qe -c input.in -p mesh.conf

In [ ]:
!cat mesh.yaml

### 7b. Using GPU4PySCF for doing superfast hessian

In [ ]:
import os
import numpy as np
import xml.etree.ElementTree as ET
from pyscf import gto

try:
    from gpu4pyscf.dft import RKS
    ON_GPU = True
except Exception:
    from pyscf.dft import RKS
    ON_GPU = False

# User settings
QE_XML = "/kaggle/working/tmp/calc.save/data-file-schema.xml"
XC = "pbe"
BASIS = "def2-tzvp"
AUXBASIS = "def2-universal-jkfit"
DO_OPT = True
OPT_STEPS = 40
OUT_PREFIX = "/kaggle/working/pyscf_hess"

BOHR = 0.529177210903
CM1_TO_KJMOL = 0.0119627


def read_qe_geometry(xml_path):
    root = ET.parse(xml_path).getroot()
    pos = root.find(".//atomic_structure/atomic_positions")
    atoms = []
    for a in pos:
        xyz = np.array([float(v) for v in a.text.split()]) * BOHR
        atoms.append((a.attrib["name"], xyz))
    return atoms


def build_mol(atoms):
    return gto.M(
        atom=[(s, tuple(x)) for s, x in atoms],
        basis=BASIS,
        unit="Angstrom",
        charge=0,
        spin=0,
        max_memory=14000,
        verbose=3,
    )


def make_mf(mol, grid):
    mf = RKS(mol, xc=XC).density_fit(auxbasis=AUXBASIS)
    mf.grids.atom_grid = grid
    mf.conv_tol = 1e-10
    mf.max_cycle = 100
    return mf


def write_xyz(mol, path, comment):
    xyz = mol.atom_coords() * BOHR
    with open(path, "w") as f:
        f.write(str(mol.natm) + "\n" + comment + "\n")
        for i in range(mol.natm):
            f.write("%-2s %16.10f %16.10f %16.10f\n" % (mol.atom_symbol(i), *xyz[i]))


def run(atoms, do_opt=DO_OPT):
    mol = build_mol(atoms)
    print("Running on GPU" if ON_GPU else "Running on CPU")
    print("Atoms", mol.natm, "basis functions", mol.nao)

    if do_opt:
        from pyscf.geomopt.geometric_solver import optimize

        mf = make_mf(mol, (75, 302))
        mol = optimize(mf, maxsteps=OPT_STEPS)
        write_xyz(mol, OUT_PREFIX + "_opt.xyz", "pyscf optimised geometry")
        print("Optimisation finished")

    mf = make_mf(mol, (99, 590))
    e = mf.kernel()
    print("SCF energy in hartree", float(e), "converged", bool(mf.converged))

    hobj = mf.Hessian()
    hess = hobj.kernel()
    if hasattr(hess, "get"):
        hess = hess.get()

    from pyscf.hessian import thermo
    res = thermo.harmonic_analysis(mol, hess, imaginary_freq=False)
    freq = np.array(res["freq_wavenumber"], dtype=float)
    freq = np.sort(freq)

    n_imag = int(np.sum(freq < -50.0))
    n_small = int(np.sum((freq < 0.0) & (freq >= -50.0)))
    zpe = 0.5 * CM1_TO_KJMOL * float(np.sum(freq[freq > 0.0]))

    print("Lowest 15 frequencies in wavenumbers", np.round(freq[:15], 1).tolist())
    print("Imaginary modes below minus 50", n_imag)
    print("Small negative modes between minus 50 and 0", n_small)
    print("Zero point energy in kJ per mol", round(zpe, 1))

    np.savez(OUT_PREFIX + ".npz", freq=freq, hess=hess,
             coords=mol.atom_coords() * BOHR,
             symbols=np.array([mol.atom_symbol(i) for i in range(mol.natm)]))
    return freq


if __name__ == "__main__":
    run(read_qe_geometry(QE_XML))

## 8. Running CEA

Find my RocketCEA and Cantera code  here: https://www.kaggle.com/code/pcsciprav/rocketcea-and-cantera

RocketCEA is a python wrapper for NASA's CEA code for finding properties in propellants and Cantera simulates gas dynamics. For those more interested in the original CEA code, check the repo here: https://github.com/nasa/cea

If you liked this notebook or found it useful, please upvote to help others also find this notebook